In [1]:
using Pkg
Pkg.activate("C:/Users/ibzja/Documents/UPF_2022_2026/4t/2n_trimestre/Practiques_tutelades/CellBasedModels.jl")
using CellBasedModels 
using GeometryBasics
using Distributions
using GLMakie, Colors
Makie.inline!(true)
using CSV, DataFrames, Statistics
using Printf, JLD2


  Activating project at `C:\Users\ibzja\Documents\UPF_2022_2026\4t\2n_trimestre\Practiques_tutelades\CellBasedModels.jl`


In [ ]:
moving_source_model = ABM(2,
    agent = Dict(
        :vx => Float64,
        :vy => Float64,
        :v => Float64,  # Swimming speed
        :theta => Float64,
        :d => Float64,
        :l => Float64,
        :m => Float64,
        :fx => Float64,
        :fy => Float64,
        :W => Float64,
        :pressure => Float64,
        :active => Bool,

        :isSource => Bool, 
        :S => Float64,

        :methyl => Float64, # Receptor methylation
        :Yp => Float64, # CheYP levels, probability of tumbling
        :G => Float64,
        :λ => Float64,
        :P => Float64,
        :M => Float64,
        :F => Float64,
        :A => Float64,
        :Ds => Float64
    ),

    model = Dict(
        :Dr_run => Float64,
        :ε0 => Float64,
        :ε1 => Float64,
        :ε2 => Float64,
        :ε3 => Float64,
        :K => Float64,
        :Nrec => Float64,
        :Ki => Float64,
        :Ka => Float64,
        :τm => Float64,
        :α => Float64,
        :ωFrec => Float64,
        :Ky => Float64,
        :Z => Float64,
        :Kz => Float64,
        :Yy => Float64,
        :DMedium => Float64,
        :delta => Float64,
    ),

    medium = Dict(
        :mm => Float64
    ),

    agentODE = quote
        # REMOVED: Manual periodic boundary wrapping (the 0.99 cushion).
        # This prevents the creation of a "dead zone" at the edges.
        # Position wrapping is now handled exclusively in agentRule to ensure 
        # agents are always on valid grid cells when sensing mm.

        # Calculate Receptor Activity based on local mm concentration
        # 'mm' is read from the grid at the current (x,y). 
        # With the cushion removed, this reads the true edge values.
        F = ε0 + ε1 * methyl + Nrec * log((1 + mm / Ki) / (1 + mm / Ka)) 
        F0 = log(((Ky * (α - K)) / (K * (Kz * Z + Yy))) - 1)        

        mx = (ε0 + Nrec * log((1 + mm / Ki) / (1 + mm / Ka)) - F0) / (- ε1)
        A = 1 / (1 + exp(F))    
        Yp = (Ky * A * α) / ((Ky * A) + (Kz * Z) + Yy)
        G = ε2 / 4 - (ε3 / 2) / (1 + (K / Yp))      

        # Position derivatives (integrated by the solver)
        dt(x) = active * vx 
        dt(y) = active * vy  
        dt(methyl) = -(1 / τm) * (methyl - mx) 
    end,

    agentRule = quote
        # Helper logic to wrap coordinates within simBox
        # We define the bounds explicitly
        xmin, xmax = simBox[1,1], simBox[1,2]
        ymin, ymax = simBox[2,1], simBox[2,2]
        width = xmax - xmin
        height = ymax - ymin

        # Function to wrap a single coordinate (Julia's mod works on floats)
        # Result is always in [min, max)
        wrap_val(val, min_val, span) = min_val + mod(val - min_val, span)

        if isSource
            active = false
            σ = sqrt(2*Ds*dt)
            
            # 1. Calculate new position with diffusion
            x_new = x + σ * randn() 
            y_new = y + σ * randn() 
            
            # 2. STRICTLY WRAP position BEFORE interacting with medium
            # This ensures the source is never in a "ghost" zone
            x = wrap_val(x_new, xmin, width)
            y = wrap_val(y_new, ymin, height)
            
            # 3. Inject chemical at the VALID wrapped location
            mm += S
        
        else
            # --- Bacteria Logic ---
            
            v_run = v
            v_tumble = 0.25 
            speed = active ? v_run : v_tumble

            Dr_tumble = 6.2      
            Dr_total = active ? Dr_run : Dr_tumble

            # 1. Determine Tumbling/Running State
            if active 
                λ = ωFrec*exp(-G)
                P_rt = 1 - exp(-λ * dt)
                
                if rand() < P_rt
                    # Switch to Tumble
                    active = false
                    # During tumble, velocity magnitude is v_tumble, direction randomizes
                    # We update theta immediately to randomize direction for the next run
                    theta += sqrt(2 * Dr_total * dt) * randn()
                    # Set velocity components for the tumble phase (isotropic or fixed speed)
                    # Your original code set vx=speed, vy=speed which implies diagonal movement?
                    # Usually tumble implies no net displacement or random reorientation.
                    # Keeping your original logic:
                    vx = speed
                    vy = speed
                else
                    # Keep Running
                    active = true
                    # Update velocity vector based on current theta
                    vx = speed * cos(theta)
                    vy = speed * sin(theta)
                    # Rotational diffusion during run
                    theta += sqrt(2 * Dr_total * dt) * randn()
                end
                
            else
                # Currently Tumbling
                λ = ωFrec*exp(G) # Rate to switch back to Run
                P_tr = 1 - exp(-λ * dt)
                
                if rand() < P_tr
                    # Switch to Run
                    active = true
                    # Reorient randomly upon switching to run (or use current theta)
                    theta += sqrt(2 * Dr_total * dt) * randn()
                    vx = speed * cos(theta)
                    vy = speed * sin(theta)
                else
                    # Keep Tumbling
                    active = false
                    vx = speed
                    vy = speed
                    theta += sqrt(2 * Dr_total * dt) * randn()
                end
            end

            # 2. STRICTLY WRAP position AFTER movement logic
            # This ensures that if the ODE step pushed the agent over the edge,
            # it teleports to the other side immediately, ready for the next mm sensing.
            x = wrap_val(x, xmin, width)
            y = wrap_val(y, ymin, height)
        end
    end,

    mediumODE = quote
        if @mediumInside()
            dt(mm) = DMedium *(@∂2(1, mm)+ @∂2(2, mm)) - delta*mm 
        # elseif @mediumBorder(1,-1)
        #     mm = 0
        # elseif @mediumBorder(1,1)
        #     mm = 0
        # elseif @mediumBorder(2,1)
        #     mm = 0
        # elseif @mediumBorder(2,-1)
        #     mm = 0
        end
    end,

    agentAlg = CBMIntegrators.Heun(),
    mediumAlg=DifferentialEquations.Euler(),
    neighborsAlg=CBMNeighbours.CellLinked(cellEdge=10)
)

PARAMETERS
	x (Float64 agent)
	y (Float64 agent)
	xₘ (Float64 medium)
	yₘ (Float64 medium)
	Ds (Float64 agent)
	F (Float64 agent)
	active (Bool agent)
	methyl (Float64 agent)
	l (Float64 agent)
	S (Float64 agent)
	M (Float64 agent)
	d (Float64 agent)
	λ (Float64 agent)
	v (Float64 agent)
	isSource (Bool agent)
	A (Float64 agent)
	fx (Float64 agent)
	vx (Float64 agent)
	fy (Float64 agent)
	m (Float64 agent)
	Yp (Float64 agent)
	P (Float64 agent)
	pressure (Float64 agent)
	vy (Float64 agent)
	W (Float64 agent)
	G (Float64 agent)
	theta (Float64 agent)
	ε1 (Float64 model)
	α (Float64 model)
	Z (Float64 model)
	Dr_run (Float64 model)
	Ka (Float64 model)
	ε3 (Float64 model)
	ε0 (Float64 model)
	DMedium (Float64 model)
	delta (Float64 model)
	Ky (Float64 model)
	Kz (Float64 model)
	K (Float64 model)
	ε2 (Float64 model)
	Nrec (Float64 model)
	τm (Float64 model)
	Yy (Float64 model)
	Ki (Float64 model)
	ωFrec (Float64 model)
	mm (Float64 medium)


UPDATE RULES
mediumODE
 if @mediumInside()
    

In [3]:
com = Community(
    moving_source_model,
    N=21,
    dt=0.01,
    simBox = [-100.0 100.0; -100.0 100.0],
    NMedium = [200, 200]
)

m = 1/100
g = 1/10000
d = 1

com.Dr_run = 0.062

# com.v =  0.33   #Velocitat neutrophil (20 microm/min) 
# com.v = 20.0    #(microm/s)
com.v = 10.0
# com.v = 5.0
# com.v = 2.5
# com.v = 1

# com.Ds = 0.1
# com.Ds = 0.15
# com.Ds = 0.25
# com.Ds = 0.5
# com.Ds = 1
# com.Ds = 1.5
# com.Ds = 2
# com.Ds = 5
com.Ds = 10


# com.DMedium = 100
com.DMedium = 10
# com.DMedium = 1
# com.DMedium = 0.1
# com.DMedium = 0.01
# com.DMedium = 0.001
# com.DMedium = 0.0001

# com.DSource = 0.063
com.delta = 0.01
# com.delta = 0.1
# com.delta = 1.0
# com.delta = 10

com.ωFrec = 1.3
com.Ki = 0.0182
com.Ka = 3.0
com.Nrec = 6.0
com.ε0   = 6.0
com.ε1   = -1.0
com.ε2   = 80
com.ε3   = 80

# com.τm = 0.1
# com.τm = 0.5
com.τm = 1.0
# com.τm = 2.0
# com.τm = 5.0
# com.τm = 10.0
# com.τm = 30.0

com.α   = 6.0

com.K = 2.0 

com.Ky = 100.0
com.Kz = 10.0
com.Z = 5.0
com.Yy = 0.1

com.m = 1.        
com.d = 1.        
com.l = 3;

com.x = rand(Uniform(com.simBox[1,:]...),com.N)
com.y = rand(Uniform(com.simBox[2,:]...),com.N)
com.theta = rand(Uniform(0,2π),com.N)

com.methyl .= 0.0
com.Yp .= com.K

src = 1

com.isSource .= false
com.isSource[src] = true

com.active .= true 
com.active[src] = false   

com.S .= 0.0
# com.xs .= 0.0
# com.ys .= 0.0
# com.xs[src] = com.x[src]
# com.ys[src] = com.y[src]

# com.S[src] = 0.0025
# com.S[src] = 0.016
# com.S[src] = 0.26
com.S[src] = 2.6
# com.S[src] = 5

# com.S[src] = 0.1
# com.S[src] = 1
# com.S[src] = 10
# com.S[src] = 100
# com.S[src] = 1000



2.6

In [4]:
outfile = "boundry_prob_2.jld2"
steps = 10000

loadToPlatform!(com, preallocateAgents=21)
com.mm = zeros(Float64, com.NMedium...)

jldopen(outfile, "w") do file

    # -------------------------
    # Metadata (written once)
    # -------------------------
    meta = JLD2.Group(file, "meta")
    meta["N"] = com.N
    meta["NMedium"] = com.NMedium
    meta["steps"] = steps

    # -------------------------
    # Main loop
    # -------------------------
    for step in 1:steps
        step!(com)

        stepname = @sprintf("step_%06d", step)
        g = JLD2.Group(file, stepname)

        # Agent-level arrays (length = N)
        g["x"] = copy(com.x)
        g["y"] = copy(com.y)
        g["theta"] = copy(com.theta)
        g["l"] = copy(com.l)
        g["d"] = copy(com.d)
        g["active"] = copy(com.active)
        g["isSource"] = copy(com.isSource)

        # Medium grid (saved once per step)
        g["mm_grid"] = copy(com.mm)
    end
end


In [5]:
data = Dict{Int, Any}()
steps = 10000
jldopen("boundry_prob_2.jld2", "r") do file
    for step in 1:steps
        key = file[@sprintf("step_%06d", step)]
        data[step] = Dict(
            "x" => copy(key["x"]),
            "y" => copy(key["y"]),
            "theta" => copy(key["theta"]),
            "l" => copy(key["l"]),
            "mm_grid" => copy(key["mm_grid"])
        )
    end
end

# Setup plot
fig = Figure(size=(800, 600))
ax = Axis(fig[1, 1], aspect=1, title="Step 1", xlabel="x (μm)", ylabel="y (μm)")

# Initialize scatter and lines
scatter_plt = scatter!(ax, Float64[], Float64[], color=:red, markersize=5, label="Bacteria")
source_plt = scatter!(ax, [0.0], [0.0], color=:cyan, markersize=5, label="Source")

# Slider
slider = Slider(fig[2, 1:2], range=1:steps, startvalue=1, update_while_dragging = true)
Label(fig[2, 3], "Step: ", fontsize=14)

# Update function
function update_plot(step)
    empty!(ax)

    g = data[step]
    mm_grid = g["mm_grid"]
    mm = reshape(mm_grid, (200,200))

    hm = heatmap!(ax, 
                range(com.simBox[1,1],com.simBox[1,2],length=size(com.mm)[1]),
                range(com.simBox[2,1],com.simBox[2,2],length=size(com.mm)[1]),
                mm_grid, 
                colormap=:viridis
                    )

    # Colorbar(fig[1, 2], hm, label="mm (μM)")

    
    x = g["x"]
    y = g["y"]
    theta = g["theta"]
    l = g["l"]

    # Update scatter
    scatter_plt[1] = x
    scatter_plt[2] = y

    # Compute rod endpoints
    xs1 = x .+ l ./ 2 .* cos.(theta)
    ys1 = y .+ l ./ 2 .* sin.(theta)
    xs2 = x .- l ./ 2 .* cos.(theta)
    ys2 = y .- l ./ 2 .* sin.(theta)
    
    for i in 1:length(x)
        linesegments!(ax, [Point2f(x[i], y[i]), Point2f(xs1[i], ys1[i])], color=:red, linewidth=2)
        linesegments!(ax, [Point2f(x[i], y[i]), Point2f(xs2[i], ys2[i])], color=:red, linewidth=2)
    end

    # Update title
    ax.title = "Step $step"

    # Redraw
    autolimits!(ax)
end

# Connect slider to update
on(slider.value) do val
    update_plot(Int(val))
end

# Initialize
update_plot(1)

# Display
# display(fig)

GLMakie.activate!()
display(fig)

GLMakie.Screen(...)

In [ ]:
GC.gc()  # Force garbage collection
Base.flush(Base.stdout)  # Ensure any buffered output is written

In [ ]:
fig = Figure(size=(900, 600))
ax1 = Axis(fig[1,1], xlabel="time", ylabel="mean distance to source")

mean_dist = Float64[] 
std_dist = Float64[]

jldopen("boundry_prob_2.jld2", "r") do file 

    for step in 1:steps
        g = file[@sprintf("step_%06d", step)]

        src = 1

        xs = g["x"][1]
        ys = g["y"][1]


        other_x = g["x"][2:com.N]

        other_y = g["y"][2:com.N]

        # Lx = Ly = 200

        # dx = other_x .- xs
        # dy = other_y .- ys

        # dx .= dx .- Lx .* round.(dx ./ Lx)
        # dy .= dy .- Ly .* round.(dy ./ Ly)

        # dists = sqrt.(dx.^2 .+ dy.^2)

        dists = sqrt.((other_x .- xs).^2 .+ (other_y .- ys).^2)

        push!(mean_dist, mean(dists))
        push!(std_dist, std(dists))

    end

    time = (1:steps) * 0.01 
    slope = (steps * sum(time .* mean_dist) - sum(time) * sum(mean_dist)) / (steps * sum(time .^ 2) - sum(time)^2)
    intercept = mean(mean_dist) - slope * mean(time)

    std_dist_1 = std_dist ./ sqrt(com.N -1)

    lines!(ax1, (1:steps)*0.01, mean_dist)
    band!(ax1, (1:steps)*0.01, mean_dist .- std_dist_1, mean_dist .+ std_dist_1, color=(:purple, 0.2), label = "Standard deviation") 

    # text!(ax1, 0.05, 0.95,text =  "Slope = $(round(slope, digits=3)) μm/s", 
    #       align=(:left, :top), 
    #       fontsize=14, 
    #       color=:black)

    # sideinfo = Label(fig4[2, 1:2], "Model parameters = Box size: $(com.simBox), v: [$(com.v[1])μm/s], tm: [$(com.τm)s]", justification = :left, color = :grey)

    # lines!(ax1, time, slope .* time .+ intercept, color=:red, linestyle=:dash, label="Linear fit")
end

# GLMakie.activate!()
display(fig)

GLMakie.Screen(...)

In [9]:
save("boundry_prob_long.png", fig)

In [6]:
function run_single_simulation(n, trial, steps)
    println("Running n=$n, trial=$trial")

    # --- Initialize community ---
    com = Community(
        moving_source_model,
        N=21,
        dt=0.01,
        simBox = [-100.0 100.0; -100.0 100.0],
        NMedium = [200, 200]
    )

    # Parameters
    Dc = 10
    delta = 0.01
    Ds = 10

    com.Ds = Ds / n^2
    com.DMedium = Dc / n
    com.delta = delta * n

    com.Dr_run = 0.062
    com.v = 10.0
    com.ωFrec = 1.3
    com.Ki = 0.0182
    com.Ka = 3.0
    com.Nrec = 6.0
    com.ε0 = 6.0
    com.ε1 = -1.0
    com.ε2 = 80
    com.ε3 = 8
    com.τm = 1.0
    com.α = 6.0
    com.K = 2.0 
    com.Ky = 100.0
    com.Kz = 10.0
    com.Z = 5.0
    com.Yy = 0.1

    com.methyl .= 0.0
    com.Yp .= com.K

    com.m = 1.        
    com.d = 1.        
    com.l = 3

    com.x = 0.0
    com.y = 0.0
    com.theta = rand(Uniform(0, 2π), com.N)

    src = 1
    com.isSource .= false
    com.isSource[src] = true
    com.active .= true 
    com.active[src] = false   
    com.S .= 0.0
    com.S[src] = 2.6

    loadToPlatform!(com, preallocateAgents=21)

    outfile = @sprintf("memory_test_%d_%d.jld2", n, trial)

    # --- Run simulation + save ---
    jldopen(outfile, "w") do file

        meta = JLD2.Group(file, "meta")
        meta["N"] = com.N
        meta["NMedium"] = com.NMedium
        meta["steps"] = steps

        for step in 1:steps
            step!(com)
            if step % 1000 == 0

                stepname = @sprintf("step_%06d", step)
                g = JLD2.Group(file, stepname)

                g["x"] = copy(com.x)
                g["y"] = copy(com.y)
                g["theta"] = copy(com.theta)
                g["l"] = copy(com.l)
                g["d"] = copy(com.d)
                g["isSource"] = copy(com.isSource)

                g["DMedium"] = com.DMedium
                g["delta"] = com.delta
                g["Ds"] = com.Ds

                g["mm_grid"] = copy(com.mm)
            end
        end
    end

    # --- Explicit cleanup ---
    com = nothing
    GC.gc(true)

    return nothing
end

run_single_simulation (generic function with 1 method)

In [4]:
function run_all_simulations(ns, n_tries, steps)
    for n in ns
        for trial in 1:n_tries
            run_single_simulation(n, trial, steps)
        end
    end
end

run_all_simulations (generic function with 1 method)

In [7]:
ns = [1,2,3]
n_tries = 3
steps = 10000

run_all_simulations(ns, n_tries, steps)

Running n=1, trial=1


OutOfMemoryError: OutOfMemoryError()